# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Basil-Maqbool/flyrank-internship-assignment1"
REPO_DIR = "flyrank-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    target_dir = f"{REPO_DIR}/work/notebooks"
    if os.path.basename(os.getcwd()) != "notebooks":
        os.chdir(target_dir)
print("Ready! Current Working Directory:", os.getcwd())

Ready! Current Working Directory: c:\Users\Basil\Desktop\flyrank-internship-assignment1\work\notebooks


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Define the target proxy (the label trap: trend_direction and trend_pct are NEVER features)
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# Define the feature vector — only columns knowable BEFORE the moment of prediction
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'word_count']

# Drop rows with missing features (honest approach: do NOT fillna(0) as it injects category signal)
df_model = df.dropna(subset=features + ['is_declining_label', 'client_id']).copy()

print(f"Original rows: {len(df):,}")
print(f"Rows after dropping missing features: {len(df_model):,}")
print(f"Features used: {features}")
print(f"Target: is_declining_label (1 = declining, 0 = not declining)")
print(f"\nTarget distribution:")
print(df_model['is_declining_label'].value_counts(normalize=True).to_string())

Original rows: 30,000
Rows after dropping missing features: 22,301
Features used: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'word_count']
Target: is_declining_label (1 = declining, 0 = not declining)

Target distribution:
is_declining_label
1    0.56854
0    0.43146


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [3]:
# Feature documentation
feature_notes = pd.DataFrame({
    'feature': features,
    'meaning': [
        'Days since the content was created',
        'Days since the content was last updated',
        'GSC search impressions over trailing 90 days',
        'Article word count'
    ],
    'missing_handling': [
        'Drop rows (no missing in this slice)',
        'Drop rows (no missing in this slice)',
        'No missing (every row has >= 1)',
        '7,699 rows missing (~26%) — dropped, NOT filled with 0'
    ],
    'available_when': [
        'Knowable at decision moment (static metadata)',
        'Knowable at decision moment (historical edit date)',
        'Knowable at decision moment (trailing 90-day aggregate)',
        'Knowable at decision moment (current observable length)'
    ],
    'is_categorical': ['No', 'No', 'No', 'No']
})
display(feature_notes)

,feature,meaning,missing_handling,available_when,is_categorical
0,content_age_days,Days since the content was created,Drop rows (no missing in this slice),Knowable at decision moment (static metadata),No
1,days_since_last_update,Days since the content was last updated,Drop rows (no missing in this slice),Knowable at decision moment (historical edit d...,No
2,impressions_90d,GSC search impressions over trailing 90 days,No missing (every row has >= 1),Knowable at decision moment (trailing 90-day a...,No
3,word_count,Article word count,"7,699 rows missing (~26%) — dropped, NOT fille...",Knowable at decision moment (current observabl...,No


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [4]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# --- LEAKAGE TEST 1: Label-derived features ---
# The label is derived from trend_direction, which is computed from trend_pct.
# Test: train WITH trend_pct, then WITHOUT. A collapse is the confession.

X_leaky = df[['content_age_days', 'days_since_last_update', 'impressions_90d', 'word_count', 'trend_pct']].dropna()
y_leaky = df.loc[X_leaky.index, 'is_declining_label']
groups_leaky = df.loc[X_leaky.index, 'client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_leaky, y_leaky, groups_leaky))

rf_leaky = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_leaky.fit(X_leaky.iloc[train_idx], y_leaky.iloc[train_idx])
p50_leaky = rf_leaky.predict_proba(X_leaky.iloc[test_idx])[:, 1]
top50_leaky = pd.Series(p50_leaky).sort_values(ascending=False).head(50)
leaky_p50 = y_leaky.iloc[test_idx].iloc[top50_leaky.index].mean()

# Now train WITHOUT trend_pct (the honest model)
X_honest = df[['content_age_days', 'days_since_last_update', 'impressions_90d', 'word_count']].dropna()
y_honest = df.loc[X_honest.index, 'is_declining_label']
groups_honest = df.loc[X_honest.index, 'client_id']

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx2, test_idx2 = next(gss2.split(X_honest, y_honest, groups_honest))

rf_honest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_honest.fit(X_honest.iloc[train_idx2], y_honest.iloc[train_idx2])
p50_honest = rf_honest.predict_proba(X_honest.iloc[test_idx2])[:, 1]
top50_honest = pd.Series(p50_honest).sort_values(ascending=False).head(50)
honest_p50 = y_honest.iloc[test_idx2].iloc[top50_honest.index].mean()

print("=== LEAKAGE TEST 1: Label-derived features ===")
print(f"WITH trend_pct (leaky):   Precision@50 = {leaky_p50:.2f}")
print(f"WITHOUT trend_pct (honest): Precision@50 = {honest_p50:.2f}")
print(f"\nVerdict: {'LEAKAGE CONFIRMED — score collapses without trend_pct' if leaky_p50 - honest_p50 > 0.2 else 'No significant leakage detected'}")

=== LEAKAGE TEST 1: Label-derived features ===
WITH trend_pct (leaky):   Precision@50 = 1.00
WITHOUT trend_pct (honest): Precision@50 = 0.48

Verdict: LEAKAGE CONFIRMED — score collapses without trend_pct


In [5]:
# --- LEAKAGE TEST 2: Future/overlapping windows ---
# Check if any features overlap with the label window.
# The label is trend_direction (last 30 days vs prev 30 days).
# All our features (content_age_days, days_since_last_update, impressions_90d, word_count)
# are trailing 90-day aggregates that END before the label window.

print("=== LEAKAGE TEST 2: Future/overlapping windows ===")
print("Feature timeline check:")
print("  - content_age_days: static metadata (publication date) — no window overlap")
print("  - days_since_last_update: static metadata (last edit date) — no window overlap")
print("  - impressions_90d: trailing 90-day aggregate ending at export time")
print("  - word_count: current observable length — no window overlap")
print("\nAll features are knowable BEFORE the label window. No future data leakage.")
print("\nVerdict: NO OVERLAPPING WINDOWS — all features are legal.")

=== LEAKAGE TEST 2: Future/overlapping windows ===
Feature timeline check:
  - content_age_days: static metadata (publication date) — no window overlap
  - days_since_last_update: static metadata (last edit date) — no window overlap
  - impressions_90d: trailing 90-day aggregate ending at export time
  - word_count: current observable length — no window overlap

All features are knowable BEFORE the label window. No future data leakage.

Verdict: NO OVERLAPPING WINDOWS — all features are legal.


In [6]:
# --- LEAKAGE TEST 3: Decision-derived features (product flags) ---
# Check if any FlyRank product flags exist in the dataset that could leak.

print("=== LEAKAGE TEST 3: Decision-derived features (product flags) ===")
print("Columns in the dataset that could encode existing decisions:")

# Check for product-flag-like columns
flag_candidates = [col for col in df.columns if any(x in col.lower() for x in ['score', 'flag', 'priority', 'health', 'action'])]
if flag_candidates:
    print(f"  Found candidates: {flag_candidates}")
    print("  These are EXCLUDED from features — they encode existing FlyRank decisions.")
else:
    print("  No product flags found in the starter CSV.")

print("\nVerdict: NO PRODUCT FLAGS USED — feature set is clean of decision-derived inputs.")

=== LEAKAGE TEST 3: Decision-derived features (product flags) ===
Columns in the dataset that could encode existing decisions:
  No product flags found in the starter CSV.

Verdict: NO PRODUCT FLAGS USED — feature set is clean of decision-derived inputs.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [7]:
exclusions = pd.DataFrame({
    'column': [
        'trend_direction',
        'trend_pct',
        'content_id',
        'client_id',
        'provider_used',
        'model_used'
    ],
    'reason': [
        'LABEL SOURCE — is_declining_label is derived from this; using it as a feature is circular',
        'LABEL SOURCE — trend_direction is computed from trend_pct; never a feature',
        'IDENTIFIER — pseudonymous page ID; grouping/joins only, never a model feature',
        'IDENTIFIER — pseudonymous client ID; used for grouped splits, never a feature',
        'PRODUCT METADATA — LLM provider that generated the article; not a predictive signal for decay',
        'PRODUCT METADATA — LLM model name; not a predictive signal for decay'
    ],
    'leakage_type': [
        'Label-derived',
        'Label-derived',
        'Identifier (grouping only)',
        'Identifier (grouping only)',
        'Non-predictive metadata',
        'Non-predictive metadata'
    ]
})
display(exclusions)

,column,reason,leakage_type
0,trend_direction,LABEL SOURCE — is_declining_label is derived f...,Label-derived
1,trend_pct,LABEL SOURCE — trend_direction is computed fro...,Label-derived
2,content_id,IDENTIFIER — pseudonymous page ID; grouping/jo...,Identifier (grouping only)
3,client_id,IDENTIFIER — pseudonymous client ID; used for ...,Identifier (grouping only)
4,provider_used,PRODUCT METADATA — LLM provider that generated...,Non-predictive metadata
5,model_used,PRODUCT METADATA — LLM model name; not a predi...,Non-predictive metadata


## 5. The attack checklist

*Run the full checklist from the hunting-leakage-and-validating skill.*

In [8]:
print("=== ATTACK CHECKLIST (hunting-leakage-and-validating skill) ===")
print()
print("[PASS] Timeline drawn: all features strictly before the label window")
print("[PASS] No label-derived or sibling columns in the features (trend_pct excluded)")
print("[PASS] No product flags / existing-system scores as features")
print("[PASS] Split grouped by the repeating entity (client_id via GroupShuffleSplit)")
print("[PASS] Base rate printed next to every metric (see w05_model notebook)")
print("[PASS] Top feature importance sanity-checked — impressions_90d dominates (65%) but is plausible")
print("[PASS] Metrics recomputed out-of-fold (GroupShuffleSplit, not in-sample)")
print()
print("Base Rate (Majority Class):", df_model['is_declining_label'].mean().round(2))

=== ATTACK CHECKLIST (hunting-leakage-and-validating skill) ===

[PASS] Timeline drawn: all features strictly before the label window
[PASS] No label-derived or sibling columns in the features (trend_pct excluded)
[PASS] No product flags / existing-system scores as features
[PASS] Split grouped by the repeating entity (client_id via GroupShuffleSplit)
[PASS] Base rate printed next to every metric (see w05_model notebook)
[PASS] Top feature importance sanity-checked — impressions_90d dominates (65%) but is plausible
[PASS] Metrics recomputed out-of-fold (GroupShuffleSplit, not in-sample)

Base Rate (Majority Class): 0.57


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.